# 1.1 Raw data profiling and split

First experimentation notebook of the house-price MLOps pipeline.

**Goal:** inspect the raw dataset, document quality and structure, then create a reproducible train/test split **before** any preprocessing.

Splitting first avoids leakage: later steps (imputation, encoding, scaling) must be fit on train only.

| Artifact | Path |
|---|---|
| Input | `1-experimentation/data/data_raw.csv` |
| Train | `1-experimentation/data/data_train.csv` |
| Test | `1-experimentation/data/data_test.csv` |


## 0. Setup

In [1]:
%pip install -q pandas scikit-learn



[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip3 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from pathlib import Path

import pandas as pd
from sklearn.model_selection import train_test_split

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 120)

RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET = "price"
CONDITION_ORDER = ["Poor", "Fair", "Good", "Excellent"]

RAW_PATH = '../data/data_raw.csv'
TRAIN_PATH = '../data/data_train.csv'
TEST_PATH = '../data/data_test.csv'

## 1. Load raw data


In [15]:
df = pd.read_csv(RAW_PATH)

print(f"Rows: {df.shape[0]} | Columns: {df.shape[1]}")
df.head()


Rows: 84 | Columns: 7


,price,sqft,bedrooms,bathrooms,location,year_built,condition
0,495000,1527,2,1.5,Suburb,1956,Good
1,752000,2526,3,2.5,Downtown,1998,Excellent
2,319000,1622,2,1.5,Rural,1975,Fair
3,1210000,3102,4,3.0,Waterfront,2005,Excellent
4,462000,1835,2,2.0,Urban,1982,Good


### Data dictionary

| Column | Expected type | Role | Description |
|---|---|---|---|
| `price` | numeric | **target** | Sale price of the house |
| `sqft` | numeric | feature | Living area in square feet |
| `bedrooms` | numeric | feature | Number of bedrooms |
| `bathrooms` | numeric | feature | Number of bathrooms (can be fractional) |
| `location` | categorical | feature | Area type (`Suburb`, `Downtown`, `Rural`, `Waterfront`, `Urban`, `Mountain`) |
| `year_built` | numeric | feature | Year of construction |
| `condition` | ordinal categorical | feature | Physical condition (`Poor`, `Fair`, `Good`, `Excellent`) |


## 2. Structural profile

In [16]:
df.info()


<class 'pandas.DataFrame'>
RangeIndex: 84 entries, 0 to 83
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   price       84 non-null     int64  
 1   sqft        84 non-null     int64  
 2   bedrooms    84 non-null     int64  
 3   bathrooms   84 non-null     float64
 4   location    84 non-null     str    
 5   year_built  84 non-null     int64  
 6   condition   84 non-null     str    
dtypes: float64(1), int64(4), str(2)
memory usage: 4.7 KB


In [17]:
schema = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "non_null": df.notna().sum(),
    "nulls": df.isna().sum(),
    "null_pct": (df.isna().mean() * 100).round(2),
    "n_unique": df.nunique(dropna=False),
    "sample_values": [
        ", ".join(map(str, df[col].dropna().unique()[:6])) for col in df.columns
    ],
})
schema


,dtype,non_null,nulls,null_pct,n_unique,sample_values
price,int64,84,0,0.0,83,"495000, 752000, 319000, 1210000, 462000, 401000"
sqft,int64,84,0,0.0,67,"1527, 2526, 1622, 3102, 1835, 1742"
bedrooms,int64,84,0,0.0,4,"2, 3, 4, 5"
bathrooms,float64,84,0,0.0,8,"1.5, 2.5, 3.0, 2.0, 1.0, 4.5"
location,str,84,0,0.0,6,"Suburb, Downtown, Rural, Waterfront, Urban, Mo..."
year_built,int64,84,0,0.0,53,"1956, 1998, 1975, 2005, 1982, 1963"
condition,str,84,0,0.0,4,"Good, Excellent, Fair, Poor"


In [18]:
print(f"Duplicate rows: {df.duplicated().sum()}")
print(f"Fully empty rows: {df.isna().all(axis=1).sum()}")

numeric_cols = df.select_dtypes(include=["number"]).columns.tolist()
categorical_cols = df.select_dtypes(exclude=["number"]).columns.tolist()
print(f"numeric     : {numeric_cols}")
print(f"categorical : {categorical_cols}")


Duplicate rows: 0
Fully empty rows: 0
numeric     : ['price', 'sqft', 'bedrooms', 'bathrooms', 'year_built']
categorical : ['location', 'condition']


## 3. Univariate profile

In [9]:
df.describe(include="number").T


,count,mean,std,min,25%,50%,75%,max
price,84.0,628559.523810,359167.825027,249000.0,374250.0,511000.0,729250.0,1680000.0
sqft,84.0,2191.500000,650.017117,1350.0,1695.0,1995.0,2590.0,3850.0
bedrooms,84.0,2.857143,0.852252,2.0,2.0,3.0,3.0,5.0
bathrooms,84.0,2.190476,0.828356,1.0,1.5,2.0,2.5,4.5
year_built,84.0,1982.047619,19.501563,1947.0,1965.0,1982.5,1995.5,2019.0


In [10]:
df.describe(include="object").T


/var/folders/sr/921l0df519s9hhw6m5_nnfb40000gn/T/ipykernel_45774/3164331651.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.describe(include="object").T


,count,unique,top,freq
location,84,6,Suburb,17
condition,84,4,Good,37


In [11]:
print("Location counts")
display(df["location"].value_counts())
print("\nCondition counts")
display(df["condition"].value_counts().reindex(CONDITION_ORDER))


Location counts


location
Suburb        17
Downtown      17
Rural         17
Urban         16
Waterfront    15
Mountain       2
Name: count, dtype: int64


Condition counts


condition
Poor          6
Fair         19
Good         37
Excellent    22
Name: count, dtype: int64

## 4. Target and bivariate profile

`price` is the regression target. Tables below summarize how it relates to size, age, location and condition.


In [12]:
corr = df[numeric_cols].corr(numeric_only=True)
print("Numeric correlation with price")
display(corr[TARGET].sort_values(ascending=False).to_frame("corr_with_price"))
print("Full numeric correlation matrix")
corr


Numeric correlation with price


,corr_with_price
price,1.000000
sqft,0.981803
bathrooms,0.963959
bedrooms,0.934870
year_built,0.892736


Full numeric correlation matrix


,price,sqft,bedrooms,bathrooms,year_built
price,1.000000,0.981803,0.934870,0.963959,0.892736
sqft,0.981803,1.000000,0.939034,0.963192,0.936448
bedrooms,0.934870,0.939034,1.000000,0.917918,0.916701
bathrooms,0.963959,0.963192,0.917918,1.000000,0.932829
year_built,0.892736,0.936448,0.916701,0.932829,1.000000


In [13]:
(
    df.groupby("location")[TARGET]
    .agg(["count", "mean", "median", "std"])
    .round(0)
    .sort_values("mean", ascending=False)
)


,count,mean,median,std
location,,,,
Waterfront,15,1306000.0,1250000.0,177595.0
Mountain,2,936000.0,936000.0,65054.0
Downtown,17,666471.0,675000.0,88645.0
Urban,16,508625.0,511000.0,24838.0
Suburb,17,404235.0,401000.0,38155.0
Rural,17,293941.0,298000.0,32221.0


In [14]:
(
    df.groupby("condition")[TARGET]
    .agg(["count", "mean", "median", "std"])
    .round(0)
    .reindex(CONDITION_ORDER)
)


,count,mean,median,std
condition,,,,
Poor,6,259167.0,259500.0,8519.0
Fair,19,338000.0,335000.0,36535.0
Good,37,549054.0,520000.0,152023.0
Excellent,22,1113955.0,1180000.0,309431.0


## 5. Train / test split

Rules for this split:

1. Split the **raw table** (features + target together).
2. Use a fixed `random_state` so the split is reproducible.
3. Stratify on **price quartiles** so both sets keep a similar target distribution. That is more stable than stratifying on `location`, where `Mountain` has only two rows.

Ratio: **80% train / 20% test**.


In [19]:
price_bins = pd.qcut(df[TARGET], q=4, labels=False, duplicates="drop")

train_df, test_df = train_test_split(
    df,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=price_bins,
)

train_df = train_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print(f"Train: {train_df.shape[0]} rows ({train_df.shape[0] / len(df):.0%})")
print(f"Test : {test_df.shape[0]} rows ({test_df.shape[0] / len(df):.0%})")
print(f"Columns match raw: {list(train_df.columns) == list(df.columns)}")


Train: 67 rows (80%)
Test : 17 rows (20%)
Columns match raw: True


### Split sanity check

In [20]:
def summarize_split(split):
    return {
        "n": len(split),
        "price_mean": split[TARGET].mean(),
        "price_median": split[TARGET].median(),
        "price_std": split[TARGET].std(),
        "sqft_mean": split["sqft"].mean(),
        "year_built_mean": split["year_built"].mean(),
    }

pd.DataFrame(
    {
        "raw": summarize_split(df),
        "train": summarize_split(train_df),
        "test": summarize_split(test_df),
    }
).T.round(1)


,n,price_mean,price_median,price_std,sqft_mean,year_built_mean
raw,84.0,628559.5,511000.0,359167.8,2191.5,1982.0
train,67.0,618925.4,510000.0,345715.1,2165.8,1981.5
test,17.0,666529.4,512000.0,417443.9,2292.9,1984.4


In [21]:
loc_cmp = pd.concat(
    [
        df["location"].value_counts(normalize=True).rename("raw"),
        train_df["location"].value_counts(normalize=True).rename("train"),
        test_df["location"].value_counts(normalize=True).rename("test"),
    ],
    axis=1,
).fillna(0).round(3)

cond_cmp = pd.concat(
    [
        df["condition"].value_counts(normalize=True).rename("raw"),
        train_df["condition"].value_counts(normalize=True).rename("train"),
        test_df["condition"].value_counts(normalize=True).rename("test"),
    ],
    axis=1,
).fillna(0).round(3).reindex(CONDITION_ORDER)

print("Location mix")
display(loc_cmp)
print("Condition mix")
display(cond_cmp)


Location mix


,raw,train,test
location,,,
Suburb,0.202,0.239,0.059
Downtown,0.202,0.194,0.235
Rural,0.202,0.194,0.235
Urban,0.190,0.164,0.294
Waterfront,0.179,0.179,0.176
Mountain,0.024,0.030,0.000


Condition mix


,raw,train,test
condition,,,
Poor,0.071,0.075,0.059
Fair,0.226,0.239,0.176
Good,0.440,0.433,0.471
Excellent,0.262,0.254,0.294


## 6. Persist split artifacts

In [24]:
train_df.to_csv(TRAIN_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)

print("Saved train/test splits.")


Saved train/test splits.


## 7. Findings for later notebooks

- Dataset is small (~85 rows) and complete: no missing values and no duplicate rows in the raw file.
- `price` is right-skewed; waterfront homes sit at the top of the price range, rural / poor-condition homes at the bottom.
- `sqft`, `bathrooms`, `bedrooms` and `year_built` are positively correlated with `price`.
- `location` and `condition` are strong categorical drivers and should be encoded in preprocessing.
- `Mountain` is rare (2 rows). Do not stratify future splits on location unless rare classes are grouped.
